In [ ]:
import os
import sys
import glob
import pickle

import lmdb
import numpy as np
import torch
import matplotlib.pyplot as plt

repo_root = os.environ.get("GAP_AWARE_ROOT", os.path.abspath("../../.."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from model_gap_aware.maskfeat import MaskFeatGapFiller

In [ ]:
LMDB_PATH = os.environ.get(
    "S2_LMDB_PATH",
    os.path.join(repo_root, "sentinel", "data", "ssl4eo_consecutive_s2c.lmdb"),
)
LOG_DIR = os.path.join(repo_root, "sentinel", "lightning_logs/pretraining")
VAL_RANGE = (0.0, 1.0)

N_SAMPLES = 5
N_TIMESTEPS = 5
TARGET_T = N_TIMESTEPS // 2
RGB_CHANNELS = [2, 1, 0]  # B4 (Red), B3 (Green), B2 (Blue) — indices into the 10-band array
SEED = None

In [ ]:
ckpt_files = sorted(glob.glob(f"{LOG_DIR}/**/checkpoints/*.ckpt", recursive=True))
if not ckpt_files:
    raise FileNotFoundError(f"No checkpoints found under {LOG_DIR} — run training first.")
print("Available checkpoints:")
for p in ckpt_files:
    print(f"  {p}")

ckpt_path = ckpt_files[-1]  # most recent; override manually if needed
# ckpt_path = f"{LOG_DIR}/<run>/checkpoints/<checkpoint>.ckpt"
print(f"\nUsing: {ckpt_path}")

model = MaskFeatGapFiller.load_from_checkpoint(ckpt_path, map_location="cpu")
model.eval()
print("Model loaded.")
print("hparams:", dict(model.hparams))
PATCH_SIZE = model.patch_size

In [ ]:
try:
    if "env" in globals() and env is not None:
        env.close()
except Exception:
    pass

env = lmdb.open(
    LMDB_PATH,
    readonly=True,
    lock=False,
    readahead=False,
    meminit=False,
    subdir=False,
)
n_total = env.stat()["entries"]
if n_total == 0:
    raise RuntimeError(f"LMDB is empty: {LMDB_PATH}")
print(f"LMDB entries: {n_total}")


def _decode_value(data: bytes) -> np.ndarray:
    try:
        s2c_bytes, s2c_shape = pickle.loads(data)
        return np.frombuffer(s2c_bytes, dtype=np.float32).reshape(s2c_shape)
    except Exception:
        ndim = int(np.frombuffer(data, dtype=np.int32, count=1, offset=0)[0])
        shape = tuple(np.frombuffer(data, dtype=np.int32, count=ndim, offset=4).tolist())
        header_nbytes = 4 * (1 + ndim)
        return np.frombuffer(data, dtype=np.float32, offset=header_nbytes).reshape(shape)


def load_sample(idx: int):
    with env.begin(write=False) as txn:
        data = txn.get(str(idx).encode())
    if data is None:
        raise IndexError(f"No entry for index {idx}")
    arr = _decode_value(data)           # (T, C, H, W)
    x = torch.from_numpy(arr.copy())   # (T, C, H, W)
    mid = x.shape[0] // 2
    y = x[mid]                          # (C, H, W) — all channels
    return x, y


def robust_rgb(b4, b3, b2, pmin=2, pmax=98, lo=None, hi=None):
    rgb = np.stack([b4, b3, b2], axis=-1).astype(np.float32)
    gaps = np.any(np.isnan(rgb), axis=-1)
    if lo is None or hi is None:
        lo = np.nanpercentile(rgb, pmin)
        hi = np.nanpercentile(rgb, pmax)
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo, hi = 0.0, 2500.0
    rgb = (rgb - lo) / (hi - lo + 1e-6)
    rgb = np.clip(rgb, 0, 1)
    rgb[gaps] = 0.0
    return rgb, gaps

In [ ]:
def compute_gap_patch_mask(frame, patch_size):
    """frame: (C, H, W) or (H, W) — NaN at gaps. Returns (H, W) bool, True = patch masked."""
    if frame.ndim == 2:
        frame = frame[np.newaxis]
    C, H, W = frame.shape
    n_ph, n_pw = H // patch_size, W // patch_size
    r = frame.reshape(C, n_ph, patch_size, n_pw, patch_size)
    patch_has_nan = np.isnan(r).any(axis=(0, 2, 4))
    return np.repeat(np.repeat(patch_has_nan, patch_size, axis=0), patch_size, axis=1).astype(bool)


def patch_mask_to_pixels(patch_mask_1d, H, patch_size):
    """Convert flat patch mask (N,) to pixel-space (H, H). Assumes square image."""
    n_p = H // patch_size
    return np.repeat(np.repeat(patch_mask_1d.reshape(n_p, n_p), patch_size, axis=0), patch_size, axis=1).astype(bool)

In [ ]:
rng = np.random.default_rng(SEED)
chosen_indices = sorted(rng.choice(n_total, N_SAMPLES, replace=False).tolist())
print(f"Selected indices: {chosen_indices}")

In [ ]:
N_COLS = N_TIMESTEPS + 3  # input frames | Target | Reconstruction | Error
fig, axes = plt.subplots(2 * N_SAMPLES, N_COLS, figsize=(3.2 * N_COLS, 3.5 * 2 * N_SAMPLES), squeeze=False)

r, g, b = RGB_CHANNELS  # indices for display

for i, idx in enumerate(chosen_indices):
    row_in   = 2 * i
    row_mask = 2 * i + 1

    x, y = load_sample(idx)   # x:(T,C,H,W)  y:(C,H,W)
    s2c = x.numpy()

    with torch.no_grad():
        preds, patch_mask = model(x.unsqueeze(0))                                    # (1,C,H,W), (1,N)
        loss_val, loss_mask = model.compute_loss(preds, y.unsqueeze(0), patch_mask)  # scalar, (1,C,H,W)

    pred_np   = preds[0].numpy()         # (C, H, W)
    y_np      = y.numpy()                # (C, H, W)
    lmask_np  = loss_mask[0].numpy()     # (C, H, W)

    H, W = s2c.shape[2], s2c.shape[3]

    # Shared lo/hi from RGB channels of all input frames + target + predictions
    all_vals = []
    for t in range(N_TIMESTEPS):
        img = s2c[t]
        all_vals.append(np.stack([img[r], img[g], img[b]], axis=-1).ravel())
    all_vals.append(np.stack([y_np[r], y_np[g], y_np[b]], axis=-1).ravel())
    all_vals.append(np.stack([pred_np[r], pred_np[g], pred_np[b]], axis=-1).ravel())
    all_vals = np.concatenate(all_vals)
    all_vals = all_vals[np.isfinite(all_vals)]
    lo_shared = float(np.percentile(all_vals, 2)) if len(all_vals) > 0 else 0.0
    hi_shared = float(np.percentile(all_vals, 98)) if len(all_vals) > 0 else 2500.0
    if not np.isfinite(lo_shared) or not np.isfinite(hi_shared) or hi_shared <= lo_shared:
        lo_shared, hi_shared = 0.0, 2500.0

    # --- Input row: original frames ---
    for t in range(N_TIMESTEPS):
        ax = axes[row_in][t]
        img = s2c[t]
        rgb, gaps = robust_rgb(img[r], img[g], img[b], lo=lo_shared, hi=hi_shared)
        ax.imshow(rgb)
        frac = 100.0 * float(gaps.sum()) / float(gaps.size)
        title = f"t={t}  gaps={frac:.1f}%"
        if t == TARGET_T:
            title += "\n(target frame)"
        ax.set_title(title, fontsize=8)
        ax.axis("off")

    ax = axes[row_in][N_TIMESTEPS]
    rgb_tgt, _ = robust_rgb(y_np[r], y_np[g], y_np[b], lo=lo_shared, hi=hi_shared)
    ax.imshow(rgb_tgt)
    ax.set_title(f"Target (idx={idx})", fontsize=8)
    ax.axis("off")

    ax = axes[row_in][N_TIMESTEPS + 1]
    blended = np.where(lmask_np > 0, pred_np, np.nan_to_num(y_np, nan=0.0))
    rgb_blend, _ = robust_rgb(blended[r], blended[g], blended[b], lo=lo_shared, hi=hi_shared)
    ax.imshow(rgb_blend)
    ax.set_title("Reconstruction\n(pred in mask, GT outside)", fontsize=8)
    ax.axis("off")

    ax = axes[row_in][N_TIMESTEPS + 2]
    err = np.abs(pred_np - np.nan_to_num(y_np)).mean(axis=0)
    lmask_2d = lmask_np.mean(axis=0) > 0
    err_display = np.where(lmask_2d, err, np.nan)
    im = ax.imshow(err_display, cmap="Reds", vmin=0)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(f"Error\n{model.loss_type.upper()}={loss_val.item():.4f}", fontsize=8)
    ax.axis("off")

    # --- Masked-view row ---
    for t in range(N_TIMESTEPS):
        ax = axes[row_mask][t]
        img = s2c[t]
        if t == TARGET_T:
            # Middle frame: gap mask + random mask from the model forward pass
            pm_np = patch_mask[0].cpu().numpy()  # (N,)
            pixel_mask = patch_mask_to_pixels(pm_np, H, PATCH_SIZE)
        else:
            pixel_mask = compute_gap_patch_mask(img, PATCH_SIZE)
        rgb, _ = robust_rgb(img[r], img[g], img[b], lo=lo_shared, hi=hi_shared)
        rgb[pixel_mask] = 0.5
        ax.imshow(rgb)
        pct = 100.0 * pixel_mask.mean()
        title = f"t={t}  masked={pct:.1f}%"
        if t == TARGET_T:
            title += "\n(gap+rand)"
        ax.set_title(title, fontsize=8)
        ax.axis("off")
    for col in range(N_TIMESTEPS, N_COLS):
        axes[row_mask][col].axis("off")

plt.suptitle("Reconstruction Visualization — Sentinel-2", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()